Title: K_Means clustering
Author: Cameron Rosenthal

1. Use the Iris Dataset for Clustering

In [424]:
from lets_plot import *
LetsPlot.setup_html()
from sklearn.datasets import load_iris
import pandas as pd


# Load the dataset
iris_raw = load_iris()

# # Convert to a DataFrame

species = pd.DataFrame(data={
    "species": [iris_raw.target_names[x] for x in iris_raw.target]
})

features = pd.DataFrame(data=iris_raw.data, columns=iris_raw.feature_names)

iris = pd.concat([species, features], axis=1)

iris.rename(columns={
    "sepal length (cm)": "sepal_length",
    "sepal width (cm)": "sepal_width",
    "petal length (cm)": "petal_length",
    "petal width (cm)": "petal_width"
}, inplace=True)

ggplot(iris) + geom_point(aes(x="sepal_length", y="sepal_width", color="species"))



(a) Apply K-means clustering with K=4

In [425]:
from sklearn.preprocessing import StandardScaler

iris_data = iris.drop(["species"], axis=1)

scaler = StandardScaler() 
X_scaled = scaler.fit_transform(iris_data)

In [426]:
from sklearn.cluster import KMeans

k_means = KMeans(n_clusters=4)
labels = k_means.fit_predict(X_scaled)



(b) Compute the root mean squared error for each cluster based on its centroids.

In [427]:
import numpy as np

assigned_centroids = k_means.cluster_centers_[labels]

squared_diffs = (X_scaled - assigned_centroids) ** 2  # shape (150, 4)
rmse = np.sqrt(squared_diffs.sum(axis=1).mean())
rmse

np.float64(0.8721323998708861)

2. Compare the clustering results with the actual class labels for cluster quality validation.

In [429]:
centroids = scaler.inverse_transform(k_means.cluster_centers_)

df_centroids = pd.DataFrame({
    'x':       centroids[:, 0],
    'y':       centroids[:, 1],
    'cluster': ['0', '1', '2', '3']   # match cluster labels for same colour scale
})

assigned_centroids = k_means.cluster_centers_[labels]
unscaled_x = scaler.inverse_transform(X_scaled)

iris_data_temp = pd.DataFrame(iris_data, columns=iris_data.columns)
labels_pd = pd.DataFrame({"label": labels})
labeled_pd = pd.concat([iris_data_temp, labels_pd], axis=1)

labeled_pd = labeled_pd.astype({
    "label": 'str'
})

(
    ggplot(iris) +
    geom_point(aes(x="sepal_length", y="sepal_width", color="species")) +
    # geom_point(data=df_centroids, mapping=aes(x='x', y='y'), shape=21, size=8, stroke=2, fill='white') +
    ggtitle("Original Iris Species")
).show()


(
    ggplot(labeled_pd) +
    geom_point(aes(x="sepal_length", y="sepal_width", color="label")) +
    geom_point(data=df_centroids, mapping=aes(x='x', y='y'), shape=21, size=8, stroke=2, fill='white') +
    ggtitle("Predicted Groups")
).show()


(a) How well do clusters align with true classes?

The clusters somewhat align. There are issues due to there being only 3 classes but four clusters.

(b) Which clusters mix classes.

The classes that tend to mix the most is the setosa grouping.

(c) Why might k=4 be suboptimal?

K=4 is suboptimal because there are only three classes, but the k_means is expecting there to be 4 classes.